# PathSnap experiment notebook
Open this notebook with the project directory as the working directory and the project environment as the kernel. Run the cells in order to reproduce the five-model experiment. The saved results and model also support an immediate demo using the README commands.

In [ ]:
from pathlib import Path
import subprocess, sys
import pandas as pd
from IPython.display import display, Image, IFrame
root = Path.cwd()
assert (root / "pathsnap").is_dir(), "Open the notebook from inside the PathSnap project folder."
assert (root / "data/train").is_dir(), "Put the supplied train and student_test folders in data/."
def run(*args):
    subprocess.run([sys.executable, "-m", *args], cwd=root, check=True)

## 1. Inspect trips and build candidate features
This reads the original four file types, preserves trip grouping and derives observation-level reference labels for development.

In [ ]:
run("pathsnap.prepare", "--data", "data", "--cache", "cache", "--workers", "2")
inventory = pd.read_csv("outputs/inventory.csv", dtype={"trip":str})
display(inventory.groupby("split")[["observations","candidate_misses","gaps_over_30s"]].sum())

## 2. Train five models and reconstruct routes
The full run trains XGBoost, LightGBM, Random Forest, KNN and an HMM observation scorer. It selects by grouped CV, runs Bayesian optimisation and compares the standalone HMM and the selected classifier with Viterbi.

In [ ]:
run("pathsnap.train", "--data", "data", "--cache", "cache")
display(pd.read_csv("outputs/model_comparison.csv")[["model","cv_macro_f1","macro_f1","top1","log_loss","inference_s"]])

In [ ]:
display(pd.read_csv("outputs/bayesian_trials.csv"))
display(pd.read_csv("outputs/decoder_summary.csv"))

## 3. Explore the map and errors
The interactive map is a student-test illustration. The demonstration UI gets its predictions from the live API.

In [ ]:
run("pathsnap.visualize", "--data", "data")
display(Image("outputs/figures/decoder.png"))
display(IFrame("outputs/example_map.html", width="100%", height=500))

## 4. Demonstrate the two-service application
Start the API and UI using the two commands in README.md. After the student-test API is running, the following cell sends fresh requests for all ten trips.

In [ ]:
run("pathsnap.verify", "--api", "http://127.0.0.1:8000", "--offline", "outputs/predictions")